# EDA: Students Performance Dataset

## Цель анализа
В этом ноутбуке я провожу первичный анализ датасета **StudentsPerformance**:
- изучаю структуру данных;
- проверяю распределения баллов;
- сравниваю результаты между группами;
- ищу факторы, которые могут быть связаны с успеваемостью.

## Почему этот проект полезен
Это не просто упражнения по `pandas`, а пример базового **exploratory data analysis (EDA)**:
от чтения данных и очистки названий столбцов до агрегаций, фильтрации и формулировки выводов.

In [ ]:
import pandas as pd
import numpy as np

students = pd.read_csv(
    "https://stepik.org/media/attachments/course/4852/StudentsPerformance.csv"
)

students = students.rename(columns={
    "parental level of education": "parental_level_of_education",
    "test preparation course": "test_preparation_course",
    "math score": "math_score",
    "reading score": "reading_score",
    "writing score": "writing_score",
    "race/ethnicity": "race_ethnicity"
})

students.head()

## 1. Первичный обзор данных
Сначала проверяю размер таблицы, типы столбцов и базовые статистики.

In [ ]:
students.shape, students.dtypes

In [ ]:
students.describe(include="all")

## 2. Базовые срезы по успеваемости
Сравню средние баллы по полу и по нескольким категориальным признакам.

In [ ]:
students.groupby("gender")[["math_score", "reading_score", "writing_score"]].mean().round(2)

In [ ]:
students.groupby("race_ethnicity")[["math_score", "reading_score", "writing_score"]].mean().round(2)

In [ ]:
students.groupby("test_preparation_course")[["math_score", "reading_score", "writing_score"]].mean().round(2)

## 3. Влияние типа питания
Один из интересных признаков — `lunch`. Посмотрю, отличаются ли средние результаты у групп.

In [ ]:
students_free_reduced = students[students["lunch"] == "free/reduced"]
students_standard = students[students["lunch"] == "standard"]

pd.DataFrame({
    "free/reduced_mean": students_free_reduced[["math_score", "reading_score", "writing_score"]].mean(),
    "standard_mean": students_standard[["math_score", "reading_score", "writing_score"]].mean(),
    "free/reduced_var": students_free_reduced[["math_score", "reading_score", "writing_score"]].var(),
    "standard_var": students_standard[["math_score", "reading_score", "writing_score"]].var(),
}).round(2)

## 4. Выборки и фильтрация
Ниже — несколько типичных операций EDA: фильтрация, поиск лучших наблюдений и выбор нужных столбцов.

In [ ]:
high_writing = students.query("writing_score > writing_score.mean()")
high_writing[["gender", "parental_level_of_education", "writing_score"]].head()

In [ ]:
score_columns = [col for col in students.columns if "score" in col]
students[score_columns].head()

In [ ]:
top_math_by_gender = (
    students.sort_values(["gender", "math_score"], ascending=False)
            .groupby("gender")
            .head(5)
            [["gender", "race_ethnicity", "math_score", "reading_score", "writing_score"]]
)

top_math_by_gender

## 5. Группировки по нескольким признакам
Здесь смотрю, как меняется средний балл по математике сразу по двум категориям.

In [ ]:
mean_math_by_group = (
    students.groupby(["gender", "race_ethnicity"], as_index=False)
            .agg(mean_math_score=("math_score", "mean"))
            .sort_values("mean_math_score", ascending=False)
)

mean_math_by_group.head(10)

## Выводы

### Что удалось увидеть
1. **Разные предметы показывают разные паттерны по группам**: обычно мальчики чуть сильнее в математике, а девочки — в чтении и письме.
2. **Прохождение preparation course связано с более высокими результатами** почти по всем предметам.
3. **Группа `standard lunch` в среднем показывает более высокие баллы**, чем `free/reduced`, что может указывать на социально-экономические различия.
4. Средние значения удобно дополнять разбором дисперсий: это помогает увидеть не только уровень результатов, но и их разброс.
5. Даже на небольшом датасете простые операции `groupby`, `query`, сортировка и фильтрация уже дают полезные аналитические наблюдения.

### Что можно улучшить дальше
- добавить визуализации распределений и выбросов;
- проверить корреляции между баллами;
- построить простую baseline-модель для прогноза одного из score-признаков.